In [ ]:
import numpy as np
from scipy.stats import norm

class ObjectTracker:
    def __init__(self, process_noise=0.1, measurement_noise=0.1, initial_state_uncertainty=1.0):
        self.process_noise = process_noise
        self.measurement_noise = measurement_noise
        
        # State: [position, velocity, acceleration]
        self.state = np.zeros((3, 1))
        self.P = np.eye(3) * initial_state_uncertainty
        
        # Process noise covariance
        self.Q = np.array([[0.25*process_noise**4, 0.5*process_noise**3, 0.5*process_noise**2],
                           [0.5*process_noise**3, process_noise**2, process_noise],
                           [0.5*process_noise**2, process_noise, 1]])
        
        self.R = measurement_noise**2
        
        # Measurement matrix (we only measure position)
        self.H = np.array([[1, 0, 0]])
        
        # State transition matrix
        dt = 1.0  # Time step
        self.F = np.array([[1, dt, 0.5*dt**2],
                           [0, 1, dt],
                           [0, 0, 1]])
        
        self.direction_probability = 0.5
        self.direction_confidence = 0.5
        
        self.change_detection_threshold = 3.0
        self.change_detection_window = 10
        self.measurements_history = []

    def update(self, measurement):
        # Prediction
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        
        # Update
        y = measurement - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T / S
        
        self.state = self.state + K * y
        self.P = (np.eye(3) - K @ self.H) @ self.P
        
        # Direction inference
        velocity = self.state[1, 0]
        self.direction_probability = norm.cdf(velocity, 0, np.sqrt(self.P[1,1]))
        self.direction_confidence = 2 * abs(self.direction_probability - 0.5)
        
        # Change detection
        self.measurements_history.append(measurement)
        if len(self.measurements_history) > self.change_detection_window:
            self.measurements_history.pop(0)
            
        if len(self.measurements_history) == self.change_detection_window:
            recent_mean = np.mean(self.measurements_history[-(self.change_detection_window//2):])
            past_mean = np.mean(self.measurements_history[:self.change_detection_window//2])
            if abs(recent_mean - past_mean) > self.change_detection_threshold:
                self.reset_state()
        
        return self.get_direction(), self.direction_confidence, self.state[1, 0], self.state[2, 0]

    def get_direction(self):
        return "right" if self.direction_probability > 0.5 else "left"

    def reset_state(self):
        self.state[1:, 0] = 0  # Reset velocity and acceleration
        self.P[1:, 1:] = np.eye(2) * self.P[0, 0]  # Increase velocity and acceleration uncertainty

def simulate_object_movement(num_steps, time_step=1.0):
    position = 0
    velocity = 0
    acceleration = 0
    measurements = []
    
    for _ in range(num_steps):
        # Update acceleration, velocity, and position
        acceleration += np.random.normal(0, 0.05)
        velocity += acceleration * time_step
        position += velocity * time_step + 0.5 * acceleration * time_step**2
        
        # Add measurement noise
        measurement = position + np.random.normal(0, 0.1)
        measurements.append(measurement)
        
        # Occasionally flip direction
        if np.random.random() < 0.02:
            velocity = -velocity
            acceleration = -acceleration
    
    return measurements

# Example usage
num_steps = 1000
measurements = simulate_object_movement(num_steps)

tracker = ObjectTracker()

for measurement in measurements:
    direction, confidence, velocity, acceleration = tracker.update(measurement)
    print(f"Direction: {direction}, Confidence: {confidence:.2f}, Velocity: {velocity:.2f}, Acceleration: {acceleration:.2f}")
